In [ ]:
import os
import numpy as np
import torch
import joblib
from tqdm import notebook as tqdm
from marker_vids import smpl_opensim_markerset, all_marker_vids
from geometry import *
mkr_idx = list(set(list(smpl_opensim_markerset.values()) + list(all_marker_vids['smpl'].values())))

In [ ]:
MIADataset_parent_path = '' 
mia_repo_path   = ''
for split in ['train', 'val']:
    cand = [line.strip() for line in open(f'{mia_repo_path}/musclesinaction/ablation/datasetsplits/{split}.txt').readlines()]
    target = f'data/mia_{split}'
    os.makedirs(target, exist_ok=True)
    cnt = 0
    for fp in tqdm.tqdm(cand):
        emg  = np.load(f'{MIADataset_parent_path}/{fp}/emgvalues.npy') / 1000.
        nf   = emg.shape[0]
        qpos = torch.from_numpy(np.load(f'{MIADataset_parent_path}/{fp}/pose.npy')).view(nf, 24, 3, 1)
        fixrot = torch.Tensor([
            [-1, 0, 0],
            [0, 0, 1],
            [0, -1, 0],
        ])[None].expand(nf, -1, -1) # 1, 1, 3, 3
        qpos[:, 0] = fixrot @ qpos[:, 0]
        qpos = qpos[..., 0]
        jpos = np.load(f'{MIADataset_parent_path}/{fp}/joints3d.npy')[..., [0, 2, 1]]
        jpos[..., 0]  *= -1
        jpos[..., -1] *= -1
        mpos = np.load(f'{MIADataset_parent_path}/{fp}/verts.npy')[..., [0, 2, 1]]
        mpos[..., 0]  *= -1
        mpos[..., -1] *= -1
        floor = np.min(mpos[..., -1])
        jpos[..., -1] -= floor
        mpos[..., -1] -= floor
        mpos = mpos[:, mkr_idx]
        qpos = qpos.permute(1, 0, 2)
        qvel = estimate_angular_velocity(qpos, 1 / 10., 'aa')
        qacc = estimate_linear_velocity(qvel, 1 / 10.)
        jpos = torch.from_numpy(jpos).permute(1, 0, 2)
        jvel = estimate_linear_velocity(jpos, 1 / 10.)
        jacc = estimate_linear_velocity(jvel, 1 / 10.)
        mpos = torch.from_numpy(mpos).permute(1, 0, 2)
        mvel = estimate_linear_velocity(mpos, 1 / 10.)
        macc = estimate_linear_velocity(mvel, 1 / 10.)
        cnt += 1
        data = {}
        data['qpos'] = qpos.permute(1, 0, 2).numpy()
        data['qvel'] = qvel.permute(1, 0, 2).numpy()
        data['qacc'] = qacc.permute(1, 0, 2).numpy()
        data['mpos'] = mpos.permute(1, 0, 2).numpy()
        data['mvel'] = mvel.permute(1, 0, 2).numpy()
        data['macc'] = macc.permute(1, 0, 2).numpy()
        data['jpos'] = jpos.permute(1, 0, 2).numpy()
        data['jvel'] = jvel.permute(1, 0, 2).numpy()
        data['jacc'] = jacc.permute(1, 0, 2).numpy()
        data['path'] = f'{MIADataset_parent_path}/{fp}'
        data['etau'] = emg
        joblib.dump(data, f'{target}/{cnt}.pkl')